# Lab 11/12 — The three formats, measured

Save **one** model three ways and measure what each flattening costs and discards.
Read the [lab brief](Lab11_12.md) first. Marked on **relationships between your own
numbers**, not their magnitudes — your disk and allocator are yours.

**Before you run anything:** set your identity in the next cell. Fill every 📝 cell.
Run top to bottom, then run the final export cell and submit the two files it names.

In [1]:
ROLL_NUMBER = "202518035"      # <- your roll number, e.g. "202512345"
NAME        = "Aditya Jana"      # <- your name

# Optional: path to a real .gguf for Part 4 (an Ollama blob works). Leave "" to skip Part 4's parse.
GGUF_PATH   = ""

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."


In [2]:
# Colab: uncomment.
# !pip -q install torch safetensors

import os, sys, time, json, struct, platform, pickle, subprocess, textwrap
from pathlib import Path
import torch, torch.nn as nn
from safetensors.torch import save_file, load_file
from safetensors import safe_open
import safetensors

WORK = Path("lab11_12_work"); WORK.mkdir(exist_ok=True)
RESULTS = {"formats": {}, "stride": {}, "trust": {}, "gguf_real": {}}
IS_LINUX = sys.platform.startswith("linux")

def rss_mb():
    "Current resident memory (VmRSS). ru_maxrss is a peak and never falls, so it is useless here."
    if IS_LINUX:
        for l in open("/proc/self/status"):
            if l.startswith("VmRSS:"): return int(l.split()[1]) / 1024
    import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

def io_field(name):
    "A counter from /proc/self/io. rchar = bytes moved by read() syscalls; read_bytes = bytes off the block device."
    if not IS_LINUX: return None
    for l in open("/proc/self/io"):
        if l.startswith(name): return int(l.split()[1])
    return None

def evict(path):
    "Drop this file's pages from the page cache — rootless, Linux only. Turns the next read cold."
    if not IS_LINUX: return
    fd = os.open(path, os.O_RDONLY)
    try: os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
    finally: os.close(fd)

print("torch", torch.__version__, "| safetensors", safetensors.__version__, "| linux:", IS_LINUX)


torch 2.14.0+cpu | safetensors 0.8.0 | linux: True


In [3]:
# One model, ~200 MB of fp32 weights. Big enough that cache and copy effects are visible.
torch.manual_seed(635)
model = nn.Sequential(nn.Linear(8192, 6144), nn.Linear(6144, 2048))
sd = model.state_dict()
total_mb = sum(t.numel()*t.element_size() for t in sd.values()) / 1e6
print(f"model: {sum(t.numel() for t in sd.values())/1e6:.1f}M params, {total_mb:.0f} MB of weights")


model: 62.9M params, 252 MB of weights


---
## Part 1 — The load path

📝 **Predict, before running the next cell.** For `.pt` and `.safetensors`, which loads
faster warm than cold? Which leaves resident memory roughly equal to the file size, and
which barely moves it? Which shows bytes read from disk on a cold load? Write your
predictions and one sentence of reasoning each.


1. Both .pt and .safetensors will load much faster warm than cold as on the cold load, the file is actually fetched from the SSD. On the warm load, the OS already has those files in the page cache from the previous run so it skips disk read entirely.

2. So.pt causes resident memory (RSS) to spike by roughly the full file size (~250 MB) while .safetensors barely moves it. Because .pt uses python's pickle which unpacks every single number from the file and builds fresh tensor copies in the RAM while .safetensors used memory mapping (mmap) so it actually does not place the numbers into the RAM until we explictly compute them.

3. Both will show physical disk reads on a cold load because the data isn't in the memory yet. But .pt reads them using standard commands (rchar), whereas .safetensors issues almost no read commands, it uses memory mapping.

In [4]:
def measure_pt(sd, path):
    torch.save(sd, path)
    fmb = path.stat().st_size/1e6
    evict(path); rb0=io_field("read_bytes"); t=time.perf_counter(); torch.load(path, weights_only=True); cold=time.perf_counter()-t; cold_rb=(io_field("read_bytes")-rb0)/1e6 if rb0 is not None else None
    t=time.perf_counter(); torch.load(path, weights_only=True); warm=time.perf_counter()-t
    r0=rss_mb(); rc0=io_field("rchar"); obj=torch.load(path, weights_only=True); rss=rss_mb()-r0; rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    del obj
    return dict(file_mb=round(fmb,1), cold_s=round(cold,3), warm_s=round(warm,3),
                cold_read_bytes_mb=round(cold_rb,1) if cold_rb is not None else None,
                load_rss_delta_mb=round(rss,1), load_rchar_mb=round(rc,1) if rc is not None else None)

def measure_safetensors(sd, path):
    save_file(sd, str(path))
    fmb = path.stat().st_size/1e6
    evict(path); rb0=io_field("read_bytes"); t=time.perf_counter(); load_file(path); cold=time.perf_counter()-t; cold_rb=(io_field("read_bytes")-rb0)/1e6 if rb0 is not None else None
    t=time.perf_counter(); load_file(path); warm=time.perf_counter()-t
    r0=rss_mb(); rc0=io_field("rchar"); obj=load_file(path); rss=rss_mb()-r0; rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    del obj
    # explicit read() of the same bytes, for the rchar contrast
    rc0=io_field("rchar"); _=path.read_bytes(); read_rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    return dict(file_mb=round(fmb,1), cold_s=round(cold,3), warm_s=round(warm,3),
                cold_read_bytes_mb=round(cold_rb,1) if cold_rb is not None else None,
                load_rss_delta_mb=round(rss,1), load_rchar_mb=round(rc,1) if rc is not None else None,
                explicit_read_rchar_mb=round(read_rc,1) if read_rc is not None else None)

RESULTS["formats"]["pt"] = measure_pt(sd, WORK/"model.pt")
RESULTS["formats"]["safetensors"] = measure_safetensors(sd, WORK/"model.safetensors")
if GGUF_PATH and Path(GGUF_PATH).exists():
    RESULTS["formats"]["gguf"] = dict(file_mb=round(Path(GGUF_PATH).stat().st_size/1e6,1))

for k, v in RESULTS["formats"].items():
    print(f"{k:<12}", v)


pt           {'file_mb': 251.7, 'cold_s': 0.716, 'warm_s': 0.334, 'cold_read_bytes_mb': 252.2, 'load_rss_delta_mb': 240.0, 'load_rchar_mb': 251.7}
safetensors  {'file_mb': 251.7, 'cold_s': 0.017, 'warm_s': 0.0, 'cold_read_bytes_mb': 2.7, 'load_rss_delta_mb': 0.0, 'load_rchar_mb': 0.0, 'explicit_read_rchar_mb': 251.7}


📝 **Explain your numbers.** Warm vs cold: what did the second load skip? `.pt` vs
`.safetensors` resident memory: why does one match the file size and the other not?
And why did `.safetensors` load with `rchar ≈ 0` while an explicit `read()` of the same
file moved the whole file? Name the mechanism in each case.

*Warm loads are substantially faster than cold loads because the second read bypasses physical storage entirely: the Linux kernel retains recently accessed file pages in the RAM page cache, converting expensive disk I/O into fast memory transfers. The resident memory (RSS) delta for .pt matches the file size (~240 MB) because pickle deserializes the archive by allocating fresh RAM and reconstructing full tensor copies, whereas .safetensors registers 0.0 MB because it uses mmap to map virtual address space directly to the file without creating user-space duplicates. Finally, .safetensors yields rchar ≈ 0 while an explicit read() records the full file size because mmap lets the kernel fetch pages on demand behind the scenes instead of issuing standard read() system calls.*

---
## Part 2 — The stride tax

📝 **Predict.** Transposing a big tensor — how many bytes move? Calling `.contiguous()`
on that transpose — how should its cost scale with tensor size? Will `safetensors` save a
transposed view?

*When transposing a tensor via v = big.T, zero bytes move in physical RAM because PyTorch only updates the tensor's metadata: it swaps the shape and stride attributes to create a new view over the existing memory buffer without touching the underlying raw data. Calling .contiguous() on that transposed view scales linearly with tensor size ($O(N)$ in both runtime and memory) because it cannot reuse the existing layout; it must allocate an entirely new buffer of equal size and copy every element sequentially into standard row-major order. Finally, safetensors will strictly refuse to save the transposed view and raise a ValueError because its zero-copy memory mapping design requires tensor bytes to sit contiguously in memory so they can be written directly to disk without dynamic layout transformations.*

In [5]:
big = torch.randn(8192, 8192)
v = big.T
RESULTS["stride"]["transpose_same_ptr"] = bool(v.data_ptr() == big.data_ptr())

sizes, times = [], []
for n in (2048, 4096, 6144, 8192):
    a = torch.randn(n, n)
    t = time.perf_counter(); _ = a.T.contiguous(); dt = (time.perf_counter()-t)*1e3
    sizes.append(round(n*n*4/1e6, 1)); times.append(round(dt, 2))
RESULTS["stride"]["contig_sizes_mb"] = sizes
RESULTS["stride"]["contig_times_ms"] = times

try:
    save_file({"w": big.T}, str(WORK/"probe.safetensors")); refused = False
except Exception: refused = True
save_file({"w": big.contiguous()}, str(WORK/"probe.safetensors"))   # the packed copy is accepted
RESULTS["stride"]["safetensors_refused_view"] = refused

print("transpose shares buffer:", RESULTS["stride"]["transpose_same_ptr"])
print("contiguous cost (MB -> ms):", list(zip(sizes, times)))
print("safetensors refused the view:", refused)


transpose shares buffer: True
contiguous cost (MB -> ms): [(16.8, 34.49), (67.1, 149.35), (151.0, 278.56), (268.4, 436.03)]
safetensors refused the view: True


📝 **Explain.** Who paid for contiguity — the machine saving the model, or the machine
loading it? Why is that the right place to put the cost for a file loaded far more often
than it is written?

*The machine saving the model paid the cost of contiguity. Because safetensors strictly refuses non-contiguous views during serialization, the saving machine is forced to perform the $O(N)$ allocation and memory copy before writing the bytes to disk. Training or exporting a model happens only once, but inference engines or distributed workers load those weights millions of times across production clusters. By paying for the contiguity upfront during the single export step, every downstream consumer gets zero-copy mmap loading with near-zero latency, eliminating redundant heap allocations and overhead on all future reads.*

---
## Part 3 — The trust boundary

📝 **Predict.** A pickle whose `__reduce__` runs code — will `torch.load` fire it under the
2.6 default? Under `weights_only=False`?

*Under the PyTorch 2.6 default (weights_only=True), torch.load will not fire the payload; it will block execution and raise an exception (pickle.UnpicklingError or UnsupportedGlobalException) because the deserializer uses an allowlist restricted exclusively to safe tensor constructs and metadata. Under weights_only=False, torch.load will fire the code immediately upon unpickling because standard Python pickle executes whatever callable is returned by the object's __reduce__ method to reconstruct state, allowing arbitrary command execution.*

In [6]:
PWNED = WORK/"PWNED.txt"

class Benign:
    "Harmless: __reduce__ names a callable; loading calls it. Swap os.system for the real thing."
    def __reduce__(self):
        return (os.system, (f'echo "payload ran" > {PWNED}',))

torch.save({"w": torch.zeros(2), "x": Benign()}, WORK/"evil.pt")

PWNED.unlink(missing_ok=True)
try:
    torch.load(WORK/"evil.pt", weights_only=True); fired_default = PWNED.exists(); blocked = False
except Exception:
    fired_default = PWNED.exists(); blocked = True

PWNED.unlink(missing_ok=True)
torch.load(WORK/"evil.pt", weights_only=False); fired_unsafe = PWNED.exists()

RESULTS["trust"] = dict(payload_fired_default=bool(fired_default),
                        blocked_by_weights_only=bool(blocked),
                        payload_fired_unsafe=bool(fired_unsafe))
print(RESULTS["trust"])


{'payload_fired_default': False, 'blocked_by_weights_only': True, 'payload_fired_unsafe': True}


📝 **The paragraph that carries this part.** `safetensors` removes the execution
mechanism. Name **at least two** things a `.safetensors` checkpoint still cannot protect
you from, and say why the format has no opinion on them.

*While safetensors completely eliminates arbitrary code execution by removing the unpickling engine, it cannot protect against data-level adversarial threats, most notably Trojan or backdoor weight poisoning and out-of-distribution numeric instability (such as injecting infinite, NaN, or extreme values that cause denial-of-service or silent failure downstream). A poisoned model can maintain high benchmark accuracy while behaving maliciously on specific trigger tokens, and corrupted weights can overflow gradients or destabilize an inference pipeline. The format has no opinion on these vectors because safetensors is strictly a zero-copy binary serialization format designed for memory layout, data types, and byte offsets; it deliberately treats tensor contents as opaque numerical arrays and delegates semantic validity, model safety, and alignment verification entirely to the host application layer.*

---
## Part 4 — Read a real file

📝 If you set `GGUF_PATH`, run the parse below. Report the architecture and vocab size it
recovered, and which metadata value types your parser could not handle.

*GGUF_PATH was not provided in the environment configuration, so execution of the live binary parse was skipped. Inspecting the parser implementation reveals that it only handles a minimal subset of the official GGUF specification: UINT32 (tag 4), FLOAT32 (tag 6), STRING (tag 8), and homogeneous ARRAY (tag 9). Any metadata keys serialized with standard GGUF value types outside this set—such as UINT8 (0), INT8 (1), UINT16 (2), INT16 (3), INT32 (5), BOOL (7), UINT64 (10), INT64 (11), or FLOAT64 (12)—are caught by unhandled.add(tag) and trigger an immediate ValueError that stops key-value decoding cleanly.*

In [7]:
def parse_gguf(path):
    UINT32, FLOAT32, STRING, ARRAY = 4, 6, 8, 9
    f = open(path, "rb")
    assert f.read(4) == b"GGUF", "not a GGUF file"
    version, = struct.unpack("<I", f.read(4))
    n_tensors, n_kv = struct.unpack("<QQ", f.read(16))
    def r_str(): return f.read(struct.unpack("<Q", f.read(8))[0]).decode(errors="replace")
    unhandled = set(); meta = {}
    def r_val(tag):
        if tag == UINT32:  return struct.unpack("<I", f.read(4))[0]
        if tag == FLOAT32: return struct.unpack("<f", f.read(4))[0]
        if tag == STRING:  return r_str()
        if tag == ARRAY:
            et, n = struct.unpack("<IQ", f.read(12))
            return [r_val(et) for _ in range(n)]
        unhandled.add(tag); raise ValueError(tag)
    for _ in range(n_kv):
        k = r_str(); tag, = struct.unpack("<I", f.read(4))
        try: meta[k] = r_val(tag)
        except ValueError: break   # hit a type we do not decode; stop cleanly
    f.close()
    arch = meta.get("general.architecture")
    toks = next((v for k, v in meta.items() if k.endswith("tokens")), [])
    return dict(arch=arch, vocab_size=len(toks) if isinstance(toks, list) else None,
                n_kv_read=len(meta), unhandled_type_tags=sorted(unhandled))

if GGUF_PATH and Path(GGUF_PATH).exists():
    RESULTS["gguf_real"] = parse_gguf(GGUF_PATH)
    print(RESULTS["gguf_real"])
else:
    print("GGUF_PATH not set - Part 4 parse skipped (still answer the 📝 cell if you inspected a file elsewhere)")


GGUF_PATH not set - Part 4 parse skipped (still answer the 📝 cell if you inspected a file elsewhere)


---
## Short answers

Fill these from **your own** recorded numbers above — the grader checks they are consistent
with `RESULTS`.

In [8]:
ANSWERS = dict(
    # Part 1
    warm_faster_than_cold = True,        # True / False, for .pt
    safetensors_rss_near_zero = True,    # True / False
    # Part 2
    contiguous_cost_grows = True,        # True / False
    # Part 3
    weights_only_blocked_it = True,      # True / False
    safetensors_still_cannot = "Backdoor/trojan weight poisoning and numerical instability (NaN/inf corruption)",       # one line: two things a safetensors file cannot protect against
)
print(ANSWERS)

{'warm_faster_than_cold': True, 'safetensors_rss_near_zero': True, 'contiguous_cost_grows': True, 'weights_only_blocked_it': True, 'safetensors_still_cannot': 'Backdoor/trojan weight poisoning and numerical instability (NaN/inf corruption)'}


---
## Export — run last

In [9]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, safetensors=safetensors.__version__, linux=IS_LINUX)
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS, answers=ANSWERS)

out = Path(f"submission_lab11_12_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))

# report what was recorded
fmts = list(RESULTS["formats"])
rec = [f"formats={fmts}", f"stride keys={list(RESULTS['stride'])}",
       f"trust keys={list(RESULTS['trust'])}", f"gguf_real={'yes' if RESULTS['gguf_real'] else 'skipped'}"]
missing = [k for k, val in ANSWERS.items() if val in (None, "")]
print("wrote", out, "\n  " + "\n  ".join(rec))
print("  UNFILLED ANSWERS:", missing or "none")
assert "pt" in fmts and "safetensors" in fmts, "Part 1 must record at least .pt and .safetensors"


wrote submission_lab11_12_202518035.json 
  formats=['pt', 'safetensors']
  stride keys=['transpose_same_ptr', 'contig_sizes_mb', 'contig_times_ms', 'safetensors_refused_view']
  trust keys=['payload_fired_default', 'blocked_by_weights_only', 'payload_fired_unsafe']
  gguf_real=skipped
  UNFILLED ANSWERS: none
